# Single-Point Radar Simulation

Simulates a single reflection point at a known distance, generates a full MIMO radar frame, then extracts a 3D point cloud from the signal.

**Pipeline:** Single point → `radar.mimo()` → Range-Doppler FFT → Point cloud extraction

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[0] / "witwin-radar"))

import numpy as np
import torch
import matplotlib.pyplot as plt
from core import Radar
from sigproc import process_pc, process_rd

torch.set_default_device('cuda')

## 1. Radar Config (TI IWR1843)

In [ ]:
config = {
    "num_tx": 3, "num_rx": 4,
    "fc": 77e9,
    "slope": 60.012,
    "adc_samples": 256,
    "adc_start_time": 6,
    "sample_rate": 4400,
    "idle_time": 7,
    "ramp_end_time": 65,
    "chirp_per_frame": 128,
    "frame_per_second": 10,
    "num_doppler_bins": 128,
    "num_range_bins": 256,
    "num_angle_bins": 64,
    "power": 15,
    "tx_loc": [[0, 0, 0], [4, 0, 0], [2, 1, 0]],
    "rx_loc": [[-6, 0, 0], [-5, 0, 0], [-4, 0, 0], [-3, 0, 0]],
}
radar = Radar(config)

## 2. Define a Single Reflection Point

Point at `(0, 0, -3)` with a small velocity along Z.

In [ ]:
point = np.array([[0, 0, -3]], dtype=np.float32)
velocity = np.array([0, 0, 3.14])  # m/s along z

def location_function(t):
    """Return (intensities, positions) at time t."""
    pos = torch.tensor(point + velocity * t, dtype=torch.float32, device='cuda')
    intensity = torch.ones(pos.shape[0], dtype=torch.float32, device='cuda')
    return intensity, pos

## 3. Generate Radar Frame

In [ ]:
frame = radar.mimo(location_function, t0=0)
print(f"Frame shape: {frame.shape}  (TX, RX, chirps, ADC)")

## 4. Extract Point Cloud

In [ ]:
pc = process_pc(radar, frame)
print(f"Point cloud: {pc.shape[0]} points, columns = [x, y, z, velocity, SNR, range]")
if pc.shape[0] > 0:
    print(f"Range of first point: {pc[0, 5]:.2f} m  (expected ~3.0 m)")

rd_mag, _, ranges, velocities = process_rd(radar, frame, tx=0, rx=0)

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) One chirp waveform (real + imag)
chirp = frame[0, 0, 64, :].cpu()
axes[0].plot(chirp.real, label='Real')
axes[0].plot(chirp.imag, label='Imag')
axes[0].set_title('Beat Signal (TX0, RX0, chirp 64)')
axes[0].set_xlabel('ADC sample')
axes[0].legend()

# (b) Range-Doppler map (physical axes)
axes[1].imshow(
    rd_mag[:, :len(ranges)],
    extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]],
    aspect='auto', origin='lower', cmap='jet',
)
axes[1].set_xlabel('Range (m)')
axes[1].set_ylabel('Velocity (m/s)')
axes[1].set_title('Range-Doppler Map (TX0, RX0)')

# (c) 3D point cloud
if pc.shape[0] > 0:
    ax3 = fig.add_subplot(1, 3, 3, projection='3d')
    axes[2].remove()
    sc = ax3.scatter(pc[:, 0], pc[:, 2], pc[:, 1], c=pc[:, 4], cmap='hot', s=10)
    ax3.scatter(*point[0, [0, 2, 1]], c='blue', s=100, marker='v', label='Ground truth')
    ax3.set_xlabel('X'); ax3.set_ylabel('Z'); ax3.set_zlabel('Y')
    ax3.set_title('Point Cloud')
    ax3.legend()
else:
    axes[2].text(0.5, 0.5, 'No detections', ha='center', va='center')
    axes[2].set_title('Point Cloud')

plt.tight_layout()